In [ ]:
import sys, os
sys.path.insert(0, '../utils')

In [ ]:
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from connectome_types import DATA_BASE_PATH
from neuron_custom_features import calc_spines_features
from plot_utils import ei_palette
from neuron_custom_features import calc_basic_degrree_features
from spines_utils import filter_valid_neuron_w_spines, split_syn_mat_by_type_four
from utils import load_synapses_position_transformed, load_neurons_table
from matplotlib.lines import Line2D
from spine_pref_utils import build_subnet_spine_ratio_df, mean_input_outdegree
from figures_utils import add_panel_label, plot_3d_box
from matplotlib.ticker import PercentFormatter
from stats_corr import p_to_stars, binned_mul_plot
from scipy.stats import pearsonr
from scipy import stats as sp_stats

In [ ]:
subnetworks = ['0.5', '0.25', '0.125', '0.0625']
subnetworks_names = [f'micro_column_network/subnetworks/{s}' for s in subnetworks]
networks = ['micro_column_network']
networks.extend(subnetworks_names)

network_names = ['Micro-column network', 'Subnetwork 0.5', 'Subnetwork 0.25', 'Subnetwork 0.125', 'Subnetwork 0.0625']
cmap = sns.color_palette("tab10", n_colors=len(network_names))
red_cmap = sns.color_palette("Reds", n_colors=len(network_names))
networks

In [ ]:
cmap = sns.color_palette(["#666666", "#9D174D", "#DB2777", "#F472B6", "#FBCFE8"], n_colors=len(network_names))
cmap

In [ ]:
dfs = []; ex_dfs = []; inh_dfs = []
dfs_EE = []


for network_name in tqdm(networks):
    if network_name == 'micro_column_network':
        neuronds_df = load_neurons_table() 
    else:
        neuronds_df = pd.read_csv(f'{DATA_BASE_PATH}/{network_name}/connectome_neurons.csv', index_col=0)

    calc_basic_degrree_features(neuronds_df)
    syn_df = load_synapses_position_transformed(base_syn_table_path=f'{DATA_BASE_PATH}/{network_name}/connectome_synapses.csv')

    df, syn_with_tags = calc_spines_features(neuronds_df, syn_df,
                                            spine_table=f'{DATA_BASE_PATH}/{network_name}/spine_table.csv',
                                            spine_table_outgoing=f'{DATA_BASE_PATH}/{network_name}/spine_table_outgoing.csv')
    
    df, filtered_syn_mat, filtered_bin_mat, filtered_mapping, filtered_reverse_mapping, ex_neurons, inh_neurons = filter_valid_neuron_w_spines(df)
    
    # synapse onto spines (F4)
    neuron_clf_type = df[['root_id', 'clf_type']].set_index('root_id').to_dict(orient='index')

    EE, EI, IE, II, ex_idx, inh_idx = split_syn_mat_by_type_four(filtered_bin_mat, filtered_mapping, neuron_clf_type)
    df_EE = build_subnet_spine_ratio_df(EE, ex_idx,  ex_idx,  filtered_mapping, 'E','E', syn_with_tags, ex_neurons)
    dfs_EE.append(df_EE)

    # Sharing (F3)
    REAL_BLOCKS = {'EE': EE, 'EI': EI, 'IE': IE, 'II': II}
    BLOCK_COL_IDX = {
        'EE': ex_idx,
        'IE': ex_idx,
        'EI': inh_idx, 
        'II': inh_idx,  
    }
    all_idx = sorted(filtered_mapping.keys())
    # sharing prop
    ex_od_df, inh_od_df = mean_input_outdegree(
        REAL_BLOCKS, BLOCK_COL_IDX, filtered_mapping, df,
        full_mat=filtered_bin_mat,
        all_idx=all_idx,
    )
    ex_neurons = ex_neurons.merge(
        ex_od_df[['root_id', 'mean_input_outdegree_EE', 'mean_input_outdegree_IE', 'mean_input_outdegree_full']],
        on='root_id', how='left'
    )

    inh_neurons = inh_neurons.merge(
        inh_od_df[['root_id', 'mean_input_outdegree_EI', 'mean_input_outdegree_II', 'mean_input_outdegree_full']],
        on='root_id', how='left'
    )

    ex_neurons = ex_neurons.merge(
        df_EE[['root_id', 'outgoing_spine_ratio']],
        on='root_id', how='left'
    )

    dfs.append(df)
    ex_dfs.append(ex_neurons)
    inh_dfs.append(inh_neurons)


In [ ]:
plt.rcParams['font.size'] = 16
plt.rcParams['legend.fontsize'] = 13
plt.rcParams['xtick.labelsize'] = 13
plt.rcParams['ytick.labelsize'] = 13
plt.rcParams['font.family'] = 'Arial'

spaital_layer_fontsize = 12

In [ ]:
def plot_x_lines(ax, neurons_col_df, color='gray', linestyle='--', lw=3, alpha=0.85):
    x = neurons_col_df.pt_position_xt
    max_x = max(x); min_x = min(x)

    ax.axvline(min_x, ymin=0, ymax=1, color=color, linestyle=linestyle, lw=lw, alpha=alpha)
    ax.axvline(max_x, ymin=0, ymax=1, color=color, linestyle=linestyle, lw=lw, alpha=alpha)

def plot_column_neurons(neurons_col_df, network_color_idx=0, ax=None, s=15, alpha=0.55, line_lw=2.75):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6), dpi=100)

    point_colors = neurons_col_df['clf_type'].map(ei_palette)
    ax.scatter(
        x=neurons_col_df['pt_position_xt'], 
        y=neurons_col_df['pt_position_yt'], 
        edgecolors=point_colors, 
        facecolors='none', 
        s=s,
        alpha=alpha,
    )
    plot_x_lines(ax, neurons_col_df, color=cmap[network_color_idx], lw=line_lw, alpha=1, linestyle='-')

    ax.set_xlabel("X (μm)"); ax.set_ylabel("Y (μm)")
    ax.invert_yaxis()
    # ax.set_xlim(560, 790)
    ax.set_ylim(750, 0)


In [ ]:
def plot_fig4_corrs(ax):
    for idx, ex_df_ in enumerate(ex_dfs):
        x = ex_df_['outgoing_spine_ratio']
        y = ex_df_['mean_input_outdegree_EE']
        valid_mask = ~np.isnan(x) & ~np.isnan(y) & ~np.isinf(x) & ~np.isinf(y)
        x = x[valid_mask]
        y = y[valid_mask]
        pearson_r, p_value = pearsonr(x, y)

        slope, intercept = np.polyfit(x, y, 1)
        x_fit = np.array([np.min(x), np.max(x)])
        y_fit = slope * x_fit + intercept

        ax.plot(x_fit, y_fit, linestyle='--', linewidth=2, alpha=1,
                color=cmap[idx],            
                label=f'R={pearson_r:.2f} {p_to_stars(p_value)}')

    ax.legend(frameon=False, loc='upper left')
    ax.set_xlabel('% of output synapses on\ntarget spines')
    ax.xaxis.set_major_formatter(PercentFormatter(xmax=1.0, decimals=0, symbol=''))

    ax.set_ylabel('Shared input strength')
    ax.spines[["top", "right"]].set_visible(False)


In [ ]:
def plot_fig3(ax):
    rho_texts = []
    rho_vals = []
    for idx, df_ee in enumerate(dfs_EE):
        clean = df_ee.dropna(subset=['n_syn', 'outgoing_spine_ratio'])
        spearman_rho, p = sp_stats.spearmanr(clean['n_syn'], clean['outgoing_spine_ratio'])
        rho_txt = f'$ρ$={spearman_rho:.2f} {p_to_stars(p)}'
        rho_texts.append(rho_txt)
        rho_vals.append(spearman_rho)

    x = range(len(rho_vals))
    bars = ax.bar(x, rho_vals, color=cmap, width=0.6)

    for bar, txt in zip(bars, rho_texts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
                txt, ha='center', va='bottom', fontsize=11)

    ax.set_xticks(list(x))
    ax.set_xticklabels(['1', '0.5', '0.25', '0.125', '0.0625'])
    ax.set_xlabel('Network')
    ax.set_ylabel('Correlation of spine targeting\nwith # of synapses')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

In [ ]:
fig = plt.figure(figsize=(26, 15), dpi=600)

outer = fig.add_gridspec(
    nrows=3, ncols=3,
    width_ratios=[1.2, 1, 1],
    height_ratios=[0.16, 1.0, 1.0],
    wspace=0.25, hspace=0.35
)

# Legend row (spans full width)
ax_top_legend = fig.add_subplot(outer[0, :])
ax_top_legend.axis('off')

# -----------------------------
# Spatial (left: Column 0)
# -----------------------------
gs_spatial = outer[1:, 0].subgridspec(2, 2, wspace=0.2, hspace=0.2)

ax_sp_0 = fig.add_subplot(gs_spatial[0, 0])
ax_sp_1 = fig.add_subplot(gs_spatial[0, 1], sharex=ax_sp_0, sharey=ax_sp_0)
ax_sp_2 = fig.add_subplot(gs_spatial[1, 0], sharex=ax_sp_0, sharey=ax_sp_0)
ax_sp_3 = fig.add_subplot(gs_spatial[1, 1], sharex=ax_sp_0, sharey=ax_sp_0)
spatial_axes = [ax_sp_0, ax_sp_1, ax_sp_2, ax_sp_3]

for idx, (ax, df_) in enumerate(zip(spatial_axes, dfs[1:])):
    plot_x_lines(ax, dfs[0], color=cmap[0], linestyle='-', lw=5)
    plot_column_neurons(df_, ax=ax, network_color_idx=idx+1, s=12, alpha=0.5, line_lw=5)
    ax.set_title(network_names[idx+1])
    plot_3d_box(ax, layer_fontsize=spaital_layer_fontsize)

    if idx in [0, 1]:  # Top row
        ax.set_xlabel("")
        plt.setp(ax.get_xticklabels(), visible=False)
    if idx in [1, 3]:  # Right column
        ax.set_ylabel("")
        plt.setp(ax.get_yticklabels(), visible=False)

    if idx == 0:
        legend_handles_ei = [
            Line2D([0], [0], marker='o', linestyle='none',
                   markeredgecolor=ei_palette['E'], markerfacecolor='none', 
                   markersize=8, label='E'),
            Line2D([0], [0], marker='o', linestyle='none',
                   markeredgecolor=ei_palette['I'], markerfacecolor='none', 
                   markersize=8, label='I')
        ]
        leg1 = ax.legend(handles=legend_handles_ei, title='', frameon=False, loc='upper left', bbox_to_anchor=(-0.02, 1.0))
        ax.add_artist(leg1)

# Network legend moved to a dedicated top row
legend_handles_networks = [
    Line2D([0], [0], linestyle='-', color=cmap[idx], lw=5.0, label=network_names[idx])
    for idx in range(len(networks))
]
ax_top_legend.legend(
    handles=legend_handles_networks,
    frameon=False,
    loc='center',
    bbox_to_anchor=(0.5, 0.0), # Added bbox_to_anchor to shift legend slightly lower
    ncol=len(legend_handles_networks),
    fontsize=20
)

# -----------------------------
# Figure 2 (Middle: Column 1)
# -----------------------------
# Middle panel (Ex/Inh Partners) stacked in column 1
ax_f2_ex = fig.add_subplot(outer[1, 1])
ax_f2_inh = fig.add_subplot(outer[2, 1], sharex=ax_f2_ex) # Share X axis with the top plot

feature_x = 'ds_spine_density'
custom_bins = [0.125, 0.375, 0.625, 0.875, 1.125, 1.375]
custom_bins_centers = [0.25, 0.5, 0.75, 1.0, 1.25]

# Top Middle: Excitatory
feature_y_ex = 'num_of_ex_incoming_neurons'
binned_mul_plot(
    ex_dfs,
    x_list=[feature_x] * len(ex_dfs),
    y_list=[feature_y_ex] * len(ex_dfs),
    cmap=cmap,
    n_bins=custom_bins,
    ax=ax_f2_ex,
    names=[''] * len(ex_dfs),
    bin_amount=[],
    add_r_to_legend=True,
    add_r_parentesis=False
)
ax_f2_ex.set_ylabel('# of local Ex partners/neuron')
ax_f2_ex.set_xlabel('') # Clear xlabel since they share the bottom one
plt.setp(ax_f2_ex.get_xticklabels(), visible=False) # Hide tick labels for the top plot

# Bottom Middle: Inhibitory
feature_y_inh = 'num_of_inh_incoming_neurons'
binned_mul_plot(
    ex_dfs,
    x_list=[feature_x] * len(ex_dfs),
    y_list=[feature_y_inh] * len(ex_dfs),
    cmap=cmap,
    n_bins=custom_bins,
    ax=ax_f2_inh,
    names=[''] * len(ex_dfs),
    bin_amount=[],
    add_r_to_legend=True,
    add_r_parentesis=False
)
ax_f2_inh.set_ylabel('# of local Inh partners/neuron')
ax_f2_inh.set_xlabel('Density of spinous synapses\n(syn/μm)')

# Apply custom bins to the shared axis
for ax_tmp in [ax_f2_ex, ax_f2_inh]:
    ax_tmp.set_xticks(custom_bins_centers)
ax_f2_inh.set_xticklabels([str(b) for b in custom_bins_centers]) # Only label the bottom plot


ax_f3 = fig.add_subplot(outer[1, 2])
ax_f4 = fig.add_subplot(outer[2, 2])


plot_fig3(ax_f3)
plot_fig4_corrs(ax_f4)

# debug_letter_placement(fig)
add_panel_label(ax_sp_0, 'A', xy=(-0.05, 1.05), fontsize=22)
add_panel_label(ax_f2_ex, 'B',xy=(-0.03, 1.05), fontsize=22)
add_panel_label(ax_f2_inh, 'C', xy=(-0.03, 1.05), fontsize=22)
add_panel_label(ax_f3, 'D', xy=(-0.03, 1.05), fontsize=22)  
add_panel_label(ax_f4, 'E', xy=(-0.03, 1.05), fontsize=22)

plt.savefig('fig5.pdf', format='pdf', bbox_inches='tight')
plt.show()